In [ ]:
# Phase 2 visual sanity check: the gate's human step. Every other check counts
# things; this one confirms the artifacts are actually knees, in the plane the
# metadata claims, and pointing the same way after mirroring. Counters cannot
# catch a study that decoded cleanly into the wrong anatomy.
import glob, os, shutil, sys

GIT_SHA = '9800bdc-wip'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
SRC = src_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.prep import load_study_npz, mirror_to_canonical, mirrors_in_plane
print('knee package imported successfully | GIT_SHA', GIT_SHA)

In [ ]:
import numpy as np
import pandas as pd

# one prepped/ directory per shard kernel
npz_paths = {
    f[:-len('.npz')]: os.path.join(d, f)
    for d in sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
    for f in os.listdir(d) if f.endswith('.npz')
}
manifests = sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))
manifest = pd.concat([pd.read_csv(p) for p in manifests], ignore_index=True)
print(f'{len(manifest)} studies in the manifest, {len(npz_paths)} artifacts across '
      f'{len(manifests)} shards')

# Deliberately include unresolved studies: those are stored unmirrored, and the
# point of looking is to see what the model gets handed for the ~48% of the
# corpus whose laterality never resolved.
rng = np.random.default_rng(0)
resolved = manifest[manifest['route'].isin(['Laterality', 'ImageLaterality',
                                            'SeriesDescription', 'BodyPartExamined'])]
unknown = manifest[manifest['route'].isin(['unknown', 'conflict'])]
picks = pd.concat([
    resolved.sample(min(6, len(resolved)), random_state=0),
    unknown.sample(min(2, len(unknown)), random_state=0),
])
print(picks[['StudyInstanceUID', 'side', 'route', 'n_series_stored']].to_string())

In [ ]:
# One row per study, one column per stored series, mid-slice of each, mirrored
# to canonical exactly the way PreppedStudyDataset does it.
import matplotlib.pyplot as plt

MAX_COLS = 4
fig, axes = plt.subplots(len(picks), MAX_COLS, figsize=(3 * MAX_COLS, 3 * len(picks)))

for row, (_, study) in enumerate(picks.iterrows()):
    uid = study['StudyInstanceUID']
    series_slices, meta = load_study_npz(npz_paths[uid])
    for col in range(MAX_COLS):
        ax = axes[row, col]
        ax.set_axis_off()
        if col >= len(series_slices):
            continue
        series_uid = sorted(series_slices)[col]
        slices = series_slices[series_uid]
        sm = meta['series'][series_uid]
        # only Axial/Coronal canonicalize by an in-plane flip; a Sagittal
        # image's horizontal axis is anterior-posterior, so flipping one would
        # mirror the knee front-to-back instead of swapping sides
        flip_side = meta['side'] if mirrors_in_plane(sm['Anatomical_Plane']) else None
        original = slices[len(slices) // 2]
        mid = mirror_to_canonical(original, flip_side)
        # label what actually happened: mirror_to_canonical no-ops when the side
        # is already canonical, so "flip_side is set" is not the same as "flipped"
        was_flipped = mid is not original
        ax.imshow(mid, cmap='gray')
        ax.set_title(f"{sm['Anatomical_Plane']} fs={sm['Fluid_Sensitive']}\n"
                     f"side={meta['side']} ({meta['route']}) "
                     f"{'flipped' if was_flipped else 'as-stored'}",
                     fontsize=8)

fig.tight_layout()
fig.savefig('/kaggle/working/phase2_visual_check.png', dpi=110)
print('saved /kaggle/working/phase2_visual_check.png -- download and review by eye')

In [ ]:
# Slice-depth view of one study: mirroring is applied per slice, so a bug there
# shows up as an inconsistent series rather than a wrong-looking single image.
uid = picks.iloc[0]['StudyInstanceUID']
series_slices, meta = load_study_npz(npz_paths[uid])
series_uid = sorted(series_slices)[0]
slices = series_slices[series_uid]
plane = meta['series'][series_uid]['Anatomical_Plane']
flip_side = meta['side'] if mirrors_in_plane(plane) else None
was_flipped = mirror_to_canonical(slices[0], flip_side) is not slices[0]

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for ax, s in zip(axes.ravel(), slices):
    ax.imshow(mirror_to_canonical(s, flip_side), cmap='gray')
    ax.set_axis_off()
for ax in axes.ravel()[len(slices):]:
    ax.set_axis_off()
fig.suptitle(f'{uid[-12:]} / {series_uid[-12:]} -- {plane}, {len(slices)} stored slices, '
             f"side={meta['side']}, {'flipped' if was_flipped else 'as-stored'}")
fig.tight_layout()
fig.savefig('/kaggle/working/phase2_visual_check_depth.png', dpi=110)
print('saved /kaggle/working/phase2_visual_check_depth.png')